# 002 Create Search Tool

这是 Deep Agents 学习线的第二份 Notebook。

上一课产物：

- 建立了 Deep Agents 的心智模型
- 验证了大模型网关连通性

本课产物：

- 一个可用的 `internet_search` 工具
- 理解 Deep Agents 对工具的要求

配套官方文档：

- [Quickstart - Tools](https://docs.langchain.com/oss/python/deepagents/quickstart)

学习目标：

1. 安装 `deepagents` 和 `tavily-python`。
2. 理解 Tavily 搜索 API 的作用。
3. 创建 `internet_search` 工具函数。
4. 手动测试搜索工具，观察返回格式。
5. 理解 Deep Agents 对工具的命名和描述要求。

## 0. 安装依赖

如果还没安装，先运行：

```bash
uv pip install deepagents tavily-python
```

In [1]:
import importlib.metadata

try:
    print('deepagents', importlib.metadata.version('deepagents'))
except importlib.metadata.PackageNotFoundError:
    print('deepagents: not installed')

try:
    print('tavily-python', importlib.metadata.version('tavily-python'))
except importlib.metadata.PackageNotFoundError:
    print('tavily-python: not installed')

deepagents 0.6.8
tavily-python 0.7.26


## 1. 加载项目配置

和上一课一样，统一从 `.env` 读取配置。

In [2]:
import os
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

MODEL_CONFIG = {
    'api_key': os.getenv('OPENAI_API_KEY', 'EMPTY'),
    'model': os.getenv('OPENAI_MODEL', 'qwq'),
    'base_url': os.getenv('OPENAI_BASE_URL', 'http://192.168.102.19:8082/v1'),
}

safe_config = dict(MODEL_CONFIG)
safe_config['api_key'] = '***'
pprint(safe_config)
print('project_root:', PROJECT_ROOT)

{'api_key': '***', 'base_url': 'http://192.168.102.19:8082/v1', 'model': 'qwq'}
project_root: /home/dev/bxc/fastapi-study


## 2. Tavily 搜索 API

Deep Agents 需要一个搜索工具来收集互联网信息。

Tavily 是一个专为 AI Agent 设计的搜索 API：

```text
用户问题 -> Tavily API -> 返回相关网页摘要
```

如果用 Java 后端类比：

```text
Tavily 像一个封装好的搜索引擎 Service。
你传入查询词，它返回结构化的搜索结果。
```

Tavily API Key 从环境变量 `TAVILY_API_KEY` 读取。

如果没有 Tavily API Key，可以先注册：https://tavily.com/

或者在 `.env` 中添加：

```text
TAVILY_API_KEY=your-api-key-here
```

In [3]:
tavily_api_key = os.getenv('TAVILY_API_KEY', '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')

if tavily_api_key:
    print('TAVILY_API_KEY: 已配置')
else:
    print('TAVILY_API_KEY: 未配置')
    print('请在 .env 中添加: TAVILY_API_KEY=your-api-key-here')
    print('或者在 https://tavily.com/ 注册获取')

TAVILY_API_KEY: 已配置


## 3. 创建 internet_search 工具

Deep Agents 对工具的要求：

1. **函数名要清晰**：`internet_search` 比 `search` 更明确
2. **参数要有描述**：`query` 参数说明搜索内容
3. **返回值要结构化**：返回字符串或字典，方便 agent 理解
4. **要有 docstring**：agent 通过 docstring 理解工具用途

这里用 `TavilySearch` 封装搜索工具。

In [ ]:
from tavily import TavilyClient

def internet_search(query: str) -> str:
    """Search the internet for information about the given query.

    Use this tool when you need to find information on the internet.
    Returns a formatted string with search results.

    Args:
        query: The search query string.

    Returns:
        Formatted search results as a string.
    """
    client = TavilyClient(api_key=tavily_api_key)
    results = client.search(query=query, max_results=5)

    formatted = []
    for i, result in enumerate(results.get('results', []), 1):
        title = result.get('title', 'N/A')
        url = result.get('url', 'N/A')
        content = result.get('content', 'N/A')
        formatted.append(f'[{i}] {title}\nURL: {url}\n{content}\n')

    return '\n'.join(formatted) if formatted else 'No results found.'

## 4. 手动测试搜索工具

先手动调用一次，观察返回格式。

In [5]:
if tavily_api_key:
    result = internet_search('LangGraph 是什么')
    print(result)
else:
    print('跳过搜索测试：TAVILY_API_KEY 未配置')
    print('配置后可运行此单元格')

[1] LangGraph 深度解析：构建可靠、可控的AI Agent 框架 - 知乎专栏
URL: https://zhuanlan.zhihu.com/p/1945401093786940263
一、LangGraph 是什么？ LangGraph 是一个用于构建有状态、多角色应用的低级编排框架。它的核心理念是将Agent 工作流建模为图（Graph），其中：. 节点

[2] 一文搞懂LangChain 新利器：LangGraph 原创 - CSDN博客
URL: https://blog.csdn.net/musicml/article/details/136441895
LangGraph 是一个有用于构建有状态和多角色的Agents 应用，它并不是一个独立于Langchain 的新框架，而是基于Langchain 之上构建的一个扩展库，可以与

[3] LangChain 发布的一个重要功能：LangGraph - 知乎专栏
URL: https://zhuanlan.zhihu.com/p/681428515
LangGraph 是LangChain 最近发布的一个重要功能，宣布LangChain 进入多智能体框架领域。通过建立在LangChain 之上，LangGraph 使开发人员可以轻松创建

[4] 一文了解LangGraph是什么？——构建智能体的新一代框架 - 53AI
URL: https://www.53ai.com/news/langchain/2025051436094.html
langchain llamaindex RAGFlow coze Dify Fastgpt Bisheng Qanything MaxKB Openclaw. AI+汽车 AI+金融 AI+工业 AI+培训 AI+SaaS AI+电商 AI+医疗. 发布日期：2025-05-14 11:47:25 浏览次数： 5428. LangGraph，AI代理开发的新一代框架，重新定义智能体构建方式。 核心内容： 1. ### **一、LangGraph：重新定义AI代理开发**. from from langgraph.graph  import import StateGraph, MessagesState from from langgraph.prebu

## 5. Deep Agents 对工具的要求

Deep Agents 会自动识别工具的以下信息：

| 信息 | 来源 | 示例 |
|---|---|---|
| 工具名 | 函数名 | `internet_search` |
| 用途 | 函数 docstring | "Search the internet for information" |
| 参数 | 函数签名 + 类型注解 | `query: str` |
| 返回值 | 函数返回 | 格式化的搜索结果字符串 |

推荐写法：

```python
def internet_search(query: str) -> str:
    """Search the internet for information about the given query."""
    # 实现
```

不推荐写法：

```python
def search(q):
    # 没有 docstring，agent 不知道这个工具是干什么的
    pass
```

In [6]:
import inspect

print('工具名:', internet_search.__name__)
print('用途:', inspect.getdoc(internet_search))
print('签名:', inspect.signature(internet_search))

工具名: internet_search
用途: Search the internet for information about the given query.

Use this tool when you need to find information on the internet.
Returns a formatted string with search results.

Args:
    query: The search query string.

Returns:
    Formatted search results as a string.
签名: (query: str) -> str


## 6. 本课小结

本课完成了：

1. 安装了 `deepagents` 和 `tavily-python`
2. 创建了 `internet_search` 工具函数
3. 手动测试了搜索功能
4. 理解了 Deep Agents 对工具的要求

产出了：

- 一个可用的 `internet_search` 工具
- 理解了工具命名、docstring、参数描述的重要性

## 下一课预告

下一课要做的是：

```text
使用 create_deep_agent 创建 agent
配置 model、tools、system_prompt
```

把本课创建的 `internet_search` 工具注册到 agent 中。

## 7. 练习

请你思考后回答：

1. 为什么 Deep Agents 要求工具有清晰的 docstring？
2. 如果让你设计一个文件读写工具，你会怎么命名和写 docstring？
3. Tavily 搜索返回的结果格式是什么？agent 如何理解这些结果？